# Using a Custom ONNX Model with GeoDeep

GeoDeep's inference pipeline accepts any ONNX model that follows a compatible input/output schema. This notebook covers:
1. How GeoDeep's ONNX interface works
2. Exporting a PyTorch model to ONNX format
3. Running the custom ONNX model with `geodeep.predict()`

This is the path to using GeoDeep as a lightweight ONNX inference engine for your own trained models.

## References
- GeoDeep GitHub: https://github.com/uav4geo/GeoDeep
- ONNX docs: https://onnx.ai/
- PyTorch ONNX export: https://pytorch.org/docs/stable/onnx.html

## 1. How GeoDeep's ONNX Interface Works

GeoDeep expects an ONNX model with:
- **Input**: `[batch, channels, height, width]` — a normalized image patch (float32, values 0–1)
- **Output**: bounding boxes + scores in COCO or YOLO format

The `geodeep.predict()` function handles tiling, model inference, and coordinate mapping.
The `geodeep.detect()` function is a higher-level wrapper that also handles model download.

## 2. Export a Pretrained PyTorch Model to ONNX

Here we export a torchvision FCOS detector as an example. In practice, use your own trained model.

In [ ]:
import torch
import torchvision

# Load a pretrained detector (FCOS with ResNet-50 backbone)
model = torchvision.models.detection.fcos_resnet50_fpn(weights="DEFAULT")
model.eval()
print(f"Model: {type(model).__name__}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create a dummy input matching GeoDeep's expected patch size (e.g., 640x640)
dummy_input = torch.randn(1, 3, 640, 640)

torch.onnx.export(
    model,
    dummy_input,
    "custom_detector.onnx",
    input_names=["input"],
    output_names=["boxes", "labels", "scores"],
    dynamic_axes={
        "input": {0: "batch", 2: "height", 3: "width"},
    },
    opset_version=17,
)
print("Exported model to custom_detector.onnx")

## 3. Verify the ONNX Model

In [ ]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession("custom_detector.onnx", providers=["CPUExecutionProvider"])

print("ONNX model inputs:")
for inp in sess.get_inputs():
    print(f"  {inp.name}: {inp.shape} ({inp.type})")

print("\nONNX model outputs:")
for out in sess.get_outputs():
    print(f"  {out.name}: {out.shape} ({out.type})")

# Test inference
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
outputs = sess.run(None, {"input": dummy})
print(f"\nTest inference successful. Output shapes: {[o.shape for o in outputs]}")

## 4. Run with GeoDeep

Pass the ONNX model path directly to `geodeep.detect()` instead of a named pre-built model.

In [ ]:
import os
import urllib.request
import geodeep

image_path = "sample.tif"
if not os.path.exists(image_path):
    urllib.request.urlretrieve(
        "https://github.com/uav4geo/GeoDeep/releases/download/v0.1.0/sample.tif",
        image_path,
    )

# Note: the custom model outputs must match GeoDeep's expected schema.
# Consult the GeoDeep docs for the exact format.
# For models with different output schemas, use onnxruntime directly (as shown above)
# and map results to GeoJSON manually.

print("Custom ONNX model loaded and validated.")
print("\nTo use with GeoDeep's tiling pipeline:")
print("  geodeep.detect(input='image.tif', output='out.geojson', model='custom_detector.onnx')")
print("\nFor models with non-standard output schemas, tile and infer manually using:")
print("  geodeep.tile() + ort.InferenceSession + geodeep.to_geojson()")

## 5. Quantize the ONNX Model for Faster CPU Inference

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="custom_detector.onnx",
    model_output="custom_detector_int8.onnx",
    weight_type=QuantType.QUInt8,
)

original_size = os.path.getsize("custom_detector.onnx") / 1e6
quantized_size = os.path.getsize("custom_detector_int8.onnx") / 1e6
print(f"Original model:   {original_size:.1f} MB")
print(f"Quantized model:  {quantized_size:.1f} MB")
print(f"Size reduction:   {(1 - quantized_size / original_size) * 100:.0f}%")
print("\nQuantized INT8 models typically run 2-4x faster on CPU with <1% accuracy loss.")